In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, roc_curve, classification_report, confusion_matrix

# Модели
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

In [ ]:
# Загружаем данные
df = pd.read_csv('./creditcard.csv')
X = df.drop('Class', axis=1)
y = df['Class']
y.head()
# Сплит
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Доля миноритарного класса в Train: {np.mean(y_train == 0):.2%}")

Train: (227845, 30), Test: (56962, 30)
Доля миноритарного класса в Train: 99.83%


In [ ]:
# RandomForestClassifier
# Задаем сетку параметров
param_grid_rf = {
    'clf__n_estimators': [50, 100, 200],       # Число деревьев
    'clf__max_depth': [None, 5, 10],           # Глубина
    'clf__min_samples_leaf': [1, 2, 4]         # Мин. объектов в листе
}

# Пайплайн
pipe_rf = Pipeline([
    ('scaler', StandardScaler()), # Лесу скейлер не нужен, но для единообразия пайплайна оставим (не навредит)
    ('clf', RandomForestClassifier(class_weight='balanced', random_state=42))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_search = GridSearchCV(
    pipe_rf, param_grid_rf, cv=cv, scoring='roc_auc', n_jobs=-1, verbose=1
)
grid_search.fit(X_train, y_train)

print(f"Лучшие параметры: {grid_search.best_params_}")
print(f"Лучший ROC-AUC на CV: {grid_search.best_score_:.4f}")

In [ ]:
# HistGradientBoostingClassifier
# Задаем сетку параметров
param_grid_rf = {
    'clf__learning_rate': [0.01, 0.05, 0.1],
    'clf__max_iter': [100, 200, 300],
    'clf__max_leaf_nodes': [10, 15, 31],
    'clf__min_samples_leaf': [10, 20, 30]
}

# Пайплайн
pipe_rf = Pipeline([
    ('scaler', StandardScaler()), # Лесу скейлер не нужен, но для единообразия пайплайна оставим (не навредит)
    ('clf', HistGradientBoostingClassifier(random_state=42))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_search = GridSearchCV(
    pipe_rf, param_grid_rf, cv=cv, scoring='roc_auc', n_jobs=-1, verbose=1
)
grid_search.fit(X_train, y_train)

print(f"Лучшие параметры для HistGradientBoostingClassifier: {grid_search.best_params_}")
print(f"Лучший ROC-AUC на CV: {grid_search.best_score_:.4f}")

Ответ записываю текстом, так как запускал код в .py файле и получил ответ за 20 минут, а в колаб работало более 2-х часов
## RandomForestClassifier
    Fitting 5 folds for each of 27 candidates, totalling 135 fits
    Лучшие параметры: {'clf__max_depth': 10, 'clf__min_samples_leaf': 1, 'clf__n_estimators': 200}
    Лучший ROC-AUC на CV: 0.9831

## HistGradientBoostingClassifier

    Fitting 5 folds for each of 81 candidates, totalling 405 fits
    Лучшие параметры для HistGradientBoostingClassifier: {'clf__learning_rate': 0.01, 'clf__max_iter': 300, 'clf__max_leaf_nodes': 15, 'clf__min_samples_leaf': 20}
    Лучший ROC-AUC на CV: 0.9833

In [ ]:
models = {
    "Random Forest": Pipeline([('sc', StandardScaler()), ('clf', RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=1, class_weight='balanced', random_state=42))]),
    "HistGradientBoosting": Pipeline([('sc', StandardScaler()), ('clf', HistGradientBoostingClassifier(learning_rate=0.01, max_iter=300, max_leaf_nodes=15, min_samples_leaf=20, random_state=42))])
}

results = []

for name, model in models.items():
    auc_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='roc_auc')

    results.append({
        "Модель": name,
        "ROC-AUC (CV)": f"{auc_scores.mean():.3f} +/- {auc_scores.std():.3f}"
    })

# Вывод в красивую табличку
df_results = pd.DataFrame(results)
print(df_results)

                  Модель     ROC-AUC (CV)
    0         Random Forest  0.983 +/- 0.009
    1  HistGradientBoosting  0.983 +/- 0.006

In [ ]:
for name, model in models.items():

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    print(f"Модель: {name}")
    print(classification_report(
        y_test,
        y_pred,
        digits=3,
        zero_division=0
    ))

    y_probs = model.predict_proba(X_test)[:, 1]
    print(f"ROC-AUC: {roc_auc_score(y_test, y_probs):.3f}")

Модель: Random Forest

                precision    recall  f1-score   support
            0      1.000     0.999     1.000     56864
            1      0.675     0.847     0.751        98
    accuracy                           0.999     56962
    macro avg      0.837     0.923     0.875     56962
    weighted avg   0.999     0.999     0.999     56962
    ROC-AUC: 0.980
Модель: HistGradientBoosting

                precision    recall  f1-score   support

           0      1.000     1.000     1.000     56864
           1      0.915     0.765     0.833        98
    accuracy                          0.999     56962
    macro avg     0.957     0.883     0.917     56962
    weighted avg  0.999     0.999     0.999     56962
    ROC-AUC: 0.974

HistGradientBoosting - отработало лучше
